# Infra-Bench FM Evaluation: SatlasPretrain on Global Substations (v1)

**Purpose:** Stand up the FM evaluation harness on existing substation imagery (May 2026 fetch). Run SatlasPretrain end-to-end as the first FM, validate the pipeline, and produce a reusable harness for the other FMs (Prithvi, Clay, CROMA, etc.).

**Design choices:**
- **3-class substation classification** (`tx_substation`, `dx_substation`, `dx_substation_untyped`) — matches existing downstream task.
- **Sentinel-2 multispectral only** for v1 (bands 0–6); S1 fusion deferred to v2.
- **Linear probe + fine-tune** conditions; matches the AlphaEarth-compatible protocol.
- **Per-region and per-class F1**, plus tail-mean F1 (mean of last 5 epochs) — the KB's honest metric for catching majority-class collapse.
- **Uses existing codebase modules** (`NpyInfrastructureDataset`, `io.percentile_normalize`) so inputs match the SimCLR pretraining baseline byte-for-byte.
- **Resize to 224×224** for SatlasPretrain; native tiles are ~61×61.

Adding the next FM is one new `Backbone` subclass. Dataset, training loop, eval table all reused.


## 1. Mount Drive + check GPU


In [1]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime -> Change runtime type -> GPU.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU available: True
GPU: Tesla T4
Memory: 15.6 GB


## 2. Install dependencies


In [2]:
%%capture
!pip install satlaspretrain-models scikit-learn


## 3. Extract codebase from Drive

Same pattern as `infra_fm_multicontinent_pretrain.ipynb`. The zip lives at `<DRIVE_ROOT>/code/infra_fm_curation.zip` and contains `infra_fm_code_only/{curation,downstream,pretraining,...}`. Putting it on `sys.path` lets us import the canonical `NpyInfrastructureDataset` (which uses your `io.percentile_normalize`, your band parsing, etc.) so this notebook's inputs match the SimCLR pretraining inputs byte-for-byte.


In [3]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infra_fm_clean'
CODE_ROOT  = f'{EXTRACT_TO}/infra_fm_code_only'

if not Path(f'{CODE_ROOT}/downstream').exists():
    print('Extracting code...')
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        for member in z.namelist():
            clean_path = member.replace('\\', '/')
            target = os.path.join(EXTRACT_TO, clean_path)
            if clean_path.endswith('/'):
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    dst.write(src.read())
    print('Done.')
else:
    print('Already extracted.')

sys.path.insert(0, CODE_ROOT)
os.chdir(CODE_ROOT)

# Import the canonical dataset class + helpers.
from downstream.common.comm_datasets import NpyInfrastructureDataset
from downstream.common.utils import set_seed
print('Imports OK')


Extracting code...
Done.
Imports OK


## 4. Configuration


In [4]:
import json

DATASETS_DRIVE = f'{DRIVE_ROOT}/datasets'        # where the .zip data files live on Drive
DATASETS_LOCAL = '/content/datasets'             # extracted destination, fast local disk
OUTPUT_DIR     = f'{DRIVE_ROOT}/results/fm_eval_satlas_s2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATASETS_LOCAL, exist_ok=True)

REGIONS = [
    'central-america',
    'australia-oceania',
    'south-america',
    'africa',
    'asia',
    'europe',
    'north-america',
]

# Classification setup.
CLASS_NAMES = ['tx_substation', 'dx_substation', 'dx_substation_untyped']
CLASS_TO_IDX = {n: i for i, n in enumerate(CLASS_NAMES)}
ASSET_TYPE_MAP = {
    'energy.transmission.substation':         'tx_substation',
    'energy.distribution.substation':         'dx_substation',
    'energy.distribution.substation_untyped': 'dx_substation_untyped',
    # Legacy short names (in case any older manifests linger):
    'tx_substation':         'tx_substation',
    'dx_substation':         'dx_substation',
    'dx_substation_untyped': 'dx_substation_untyped',
}

# Training config.
BAND_INDICES = '0,1,2,3,4,5,6'   # Sentinel-2 multispectral only; S1 (7,8) deferred to v2
IMAGE_SIZE   = 224                # SatlasPretrain-friendly; resized from native ~61x61
SEED         = 42

# Per-condition hyperparameters; defaults below, override per run if needed.
LP_EPOCHS, LP_BATCH, LP_LR = 30, 32, 1e-3
FT_EPOCHS, FT_BATCH, FT_LR = 30, 16, 1e-4

set_seed(SEED)
print(f'Bands: {BAND_INDICES}')
print(f'Output dir: {OUTPUT_DIR}')


Bands: 0,1,2,3,4,5,6
Output dir: /content/drive/MyDrive/infra_fm/results/fm_eval_satlas_s2


## 5. Extract region data zips

The data on Drive lives as zip files: `<DRIVE_ROOT>/datasets/<something>.zip` per region. The cell below:
1. Lists what's actually on Drive so you can see the real filenames.
2. Extracts each region's zip to `/content/datasets/` (fast local SSD).
3. Handles two common zip layouts: **wrapped** (zip contains `dataset_<region>_stac_v1/...`) or **bare** (zip contains `manifest.json` + `images/` at root).
4. Sanity-checks the extracted layout before proceeding.

If the zip filenames don't match the pattern below, update `ZIP_NAME_FOR_REGION`.


In [5]:
import zipfile, shutil, time
from pathlib import Path

# Show what's actually on Drive.
drive_path = Path(DATASETS_DRIVE)
print(f'Contents of {DATASETS_DRIVE}:')
if drive_path.exists():
    for entry in sorted(drive_path.iterdir()):
        if entry.suffix == '.zip':
            size_gb = entry.stat().st_size / 1e9
            print(f'  {entry.name:<50s} {size_gb:>6.2f} GB')
        elif entry.is_dir():
            print(f'  {entry.name}/  (dir)')
else:
    print('  (path does not exist)')
print()

# Map region -> zip filename. Adjust if your naming differs.
def ZIP_NAME_FOR_REGION(region):
    # Try these in order; first match wins.
    candidates = [
        f'dataset_{region}_stac_v1.zip',
        f'{region}_stac_v1.zip',
        f'dataset_{region}.zip',
        f'{region}.zip',
    ]
    for name in candidates:
        if (drive_path / name).exists():
            return name
    return None


Contents of /content/drive/MyDrive/infra_fm/datasets:
  dataset_africa_stac_v1.zip                           0.18 GB
  dataset_asia_stac_v1.zip                             0.76 GB
  dataset_australia-oceania_stac_v1.zip                0.11 GB
  dataset_central-america_stac_v1.zip                  0.14 GB
  dataset_europe_stac_v1.zip                           0.88 GB
  dataset_north-america_stac_v1.zip                    1.20 GB
  dataset_south-america_stac_v1.zip                    0.39 GB



In [6]:
def extract_region(region, force=False):
    zip_name = ZIP_NAME_FOR_REGION(region)
    if zip_name is None:
        print(f'  [SKIP] {region:<22s} no zip found on Drive')
        return False

    src_zip = drive_path / zip_name
    target_dir = Path(DATASETS_LOCAL) / f'dataset_{region}_stac_v1'

    # Already extracted?
    if not force and target_dir.exists() and (target_dir / 'manifest.json').exists():
        n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
        print(f'  [DONE] {region:<22s} already extracted ({n} tiles)')
        return True

    t0 = time.time()
    print(f'  [EXTRACTING] {region:<22s} {zip_name} -> {target_dir.name} ...', end=' ', flush=True)

    # Peek at the zip to figure out layout (wrapped vs. bare).
    with zipfile.ZipFile(src_zip) as zf:
        names = zf.namelist()
        # If any top-level entry is the region directory, it's wrapped.
        wrapped_prefix = f'dataset_{region}_stac_v1/'
        is_wrapped = any(n.startswith(wrapped_prefix) for n in names)

        if is_wrapped:
            # Extract straight into DATASETS_LOCAL; the wrapper dir will land at the right place.
            zf.extractall(DATASETS_LOCAL)
        else:
            # Bare layout: extract into target_dir directly.
            target_dir.mkdir(parents=True, exist_ok=True)
            zf.extractall(target_dir)

    elapsed = time.time() - t0
    n = len(list((target_dir / 'images').glob('*.npy'))) if (target_dir / 'images').exists() else 0
    print(f'done in {elapsed:.0f}s ({n} tiles)')
    return True


print('Extracting region datasets:')
for region in REGIONS:
    extract_region(region)

# Final sanity check.
print(f'\n{"region":<22s} {"manifest":>10s} {"n_tiles_dir":>12s}')
print('-' * 50)
available = []
for region in REGIONS:
    p = Path(DATASETS_LOCAL) / f'dataset_{region}_stac_v1'
    manifest_ok = (p / 'manifest.json').exists()
    images_dir = p / 'images'
    n_tiles = len(list(images_dir.glob('*.npy'))) if images_dir.exists() else 0
    print(f'{region:<22s} {str(manifest_ok):>10s} {n_tiles:>12d}')
    if manifest_ok and n_tiles > 0:
        available.append(region)
print(f'\nReady regions: {len(available)} / {len(REGIONS)}')


Extracting region datasets:
  [EXTRACTING] central-america        dataset_central-america_stac_v1.zip -> dataset_central-america_stac_v1 ... done in 6s (3144 tiles)
  [EXTRACTING] australia-oceania      dataset_australia-oceania_stac_v1.zip -> dataset_australia-oceania_stac_v1 ... done in 5s (3257 tiles)
  [EXTRACTING] south-america          dataset_south-america_stac_v1.zip -> dataset_south-america_stac_v1 ... done in 18s (11313 tiles)
  [EXTRACTING] africa                 dataset_africa_stac_v1.zip -> dataset_africa_stac_v1 ... done in 10s (5869 tiles)
  [EXTRACTING] asia                   dataset_asia_stac_v1.zip -> dataset_asia_stac_v1 ... done in 32s (18441 tiles)
  [EXTRACTING] europe                 dataset_europe_stac_v1.zip -> dataset_europe_stac_v1 ... done in 33s (17622 tiles)
  [EXTRACTING] north-america          dataset_north-america_stac_v1.zip -> dataset_north-america_stac_v1 ... done in 50s (24659 tiles)

region                   manifest  n_tiles_dir
------------------

**Stop here if any region you want included shows `False` / `0` above.** Common fixes:
- Wrong path: edit `DATASETS_DRIVE`.
- Wrong zip filename pattern: extend `ZIP_NAME_FOR_REGION`'s `candidates` list.
- Wrong internal structure: the extraction logic handles wrapped vs. bare, but a third layout (e.g., nested differently) needs manual handling.

If a zip is genuinely missing from Drive (still being uploaded, lost), proceed with the regions you have — the rest of the notebook gracefully skips missing ones.


## 6. Build datasets

Uses your canonical `NpyInfrastructureDataset` from `downstream.common.comm_datasets`. That class handles:
- Manifest parsing (the `{"records": [...]}` shape).
- Band selection (`band_indices='0,1,2,3,4,5,6'` = S2 MS only).
- Percentile normalization (matches `io.percentile_normalize`).
- HWC/CHW handling, min-size filtering, the works.

A thin wrapper (`SubstationLabelWrapper`) adds the 3-class label mapping that `NpyInfrastructureDataset` doesn't do natively (it returns `asset_type` as a string, we need int labels), plus the resize-to-224 step.


In [7]:
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, ConcatDataset


class SubstationLabelWrapper(Dataset):
    """Adds label mapping + resize on top of NpyInfrastructureDataset.

    The base dataset returns:
        {'image': tensor(C, H, W), 'asset_type': str, 'asset_id': str, ...}

    This wrapper:
      - drops samples whose asset_type isn't in ASSET_TYPE_MAP
      - maps asset_type -> integer label (0/1/2)
      - resizes (C, H, W) -> (C, INPUT_SIZE, INPUT_SIZE) via bilinear interpolation
      - attaches the region name so per-region evaluation works
    """
    def __init__(self, base_dataset, region, input_size=IMAGE_SIZE):
        self.base = base_dataset
        self.region = region
        self.input_size = input_size

        # Pre-filter to only the records that map to a valid label.
        self.valid_indices = []
        self.labels = []
        for i, rec in enumerate(base_dataset.records):
            asset_type = rec.asset_type
            mapped = ASSET_TYPE_MAP.get(asset_type)
            if mapped is None:
                continue
            self.valid_indices.append(i)
            self.labels.append(CLASS_TO_IDX[mapped])

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        base_idx = self.valid_indices[idx]
        sample = self.base[base_idx]
        img = sample['image']                        # (C, H, W) float32, percentile-normalized
        # Resize to fixed size.
        img = F.interpolate(img.unsqueeze(0), size=(self.input_size, self.input_size),
                            mode='bilinear', align_corners=False).squeeze(0)
        return {
            'image':    img,
            'label':    self.labels[idx],
            'asset_id': sample.get('asset_id', ''),
            'region':   self.region,
        }


def build_region_dataset(region):
    """Load one region via NpyInfrastructureDataset + label wrapper."""
    dataset_path = f'{DATASETS_LOCAL}/dataset_{region}_stac_v1'
    base = NpyInfrastructureDataset(
        dataset_root=dataset_path,
        band_indices=BAND_INDICES,
        require_labels=True,
        allowed_asset_types=list(ASSET_TYPE_MAP.keys()),
    )
    return SubstationLabelWrapper(base, region=region)


region_datasets = {}
for region in available:
    try:
        ds = build_region_dataset(region)
        if len(ds) == 0:
            print(f'  {region:<22s} 0 valid samples (skipped)')
            continue
        region_datasets[region] = ds
        # Show class distribution.
        from collections import Counter
        counts = Counter(ds.labels)
        labeled = {CLASS_NAMES[c]: n for c, n in counts.items()}
        print(f'  {region:<22s} n={len(ds):>7d}  {labeled}')
    except Exception as e:
        print(f'  {region:<22s} FAILED: {e}')


  central-america        n=   1361  {'dx_substation': 148, 'dx_substation_untyped': 881, 'tx_substation': 332}
  australia-oceania      n=   3257  {'dx_substation': 747, 'dx_substation_untyped': 2135, 'tx_substation': 375}
  south-america          n=  11313  {'dx_substation': 3282, 'dx_substation_untyped': 6789, 'tx_substation': 1242}
  africa                 n=   5869  {'dx_substation': 1020, 'dx_substation_untyped': 3670, 'tx_substation': 1179}
  asia                   n=  18441  {'dx_substation': 6588, 'dx_substation_untyped': 11823, 'tx_substation': 30}
  europe                 n=  17622  {'dx_substation_untyped': 16068, 'dx_substation': 1197, 'tx_substation': 357}
  north-america          n=  24659  {'dx_substation': 7768, 'dx_substation_untyped': 14262, 'tx_substation': 2629}


In [8]:
# Inspect one sample to confirm shape, dtype, value range.
first_region = next(iter(region_datasets))
sample = region_datasets[first_region][0]
img = sample['image']
print(f"Region: {sample['region']}")
print(f"Image:  shape={tuple(img.shape)}, dtype={img.dtype}")
print(f"        min={img.min().item():.4f}, max={img.max().item():.4f}, mean={img.mean().item():.4f}")
print(f"Label:  {sample['label']} ({CLASS_NAMES[sample['label']]})")
print(f"Asset:  {sample['asset_id']}")


Region: central-america
Image:  shape=(7, 224, 224), dtype=torch.float32
        min=0.0000, max=1.0000, mean=0.4686
Label:  1 (dx_substation)
Asset:  osm_way_48655527


/content/infra_fm_clean/infra_fm_code_only/downstream/common/comm_datasets.py:201: UserWarning: Tile has 10 bands but band_indices=[0, 1, 2, 3, 4, 5, 6] selects only 7. Pass --band-indices explicitly to use all bands (e.g. 0,1,2,3,4,5,6 for sentinel2_ms).
  image = self._load_image(record.path)


Expected: `shape=(7, 224, 224)`, `dtype=torch.float32`, `min~0.0`, `max~1.0`. If values are way outside `[0, 1]`, percentile normalization isn't being applied — indicates the codebase import path is wrong or `NpyInfrastructureDataset` was loaded from somewhere unexpected.


## 7. Train / val / test splits

Stratified by class within each region. Per-region splits preserved so per-region F1 is meaningful.


In [9]:
import random

TRAIN_FRAC, VAL_FRAC = 0.7, 0.15  # remainder -> test


def stratified_split(dataset, seed=SEED):
    by_class = {}
    for i, label in enumerate(dataset.labels):
        by_class.setdefault(label, []).append(i)
    rng = random.Random(seed)
    train, val, test = [], [], []
    for cls in sorted(by_class):
        idxs = by_class[cls].copy()
        rng.shuffle(idxs)
        n = len(idxs)
        n_train = int(n * TRAIN_FRAC)
        n_val   = int(n * VAL_FRAC)
        train.extend(idxs[:n_train])
        val.extend(idxs[n_train:n_train + n_val])
        test.extend(idxs[n_train + n_val:])
    return train, val, test


class SubsetView(Dataset):
    def __init__(self, base, indices):
        self.base = base
        self.indices = indices
        self.region = base.region
    def __len__(self): return len(self.indices)
    def __getitem__(self, i): return self.base[self.indices[i]]


splits = {}
for region, ds in region_datasets.items():
    tr, va, te = stratified_split(ds)
    splits[region] = {
        'train': SubsetView(ds, tr),
        'val':   SubsetView(ds, va),
        'test':  SubsetView(ds, te),
    }
    print(f'  {region:<22s} train={len(tr):>7d} val={len(va):>6d} test={len(te):>6d}')

train_global = ConcatDataset([s['train'] for s in splits.values()])
val_global   = ConcatDataset([s['val']   for s in splits.values()])
test_global  = ConcatDataset([s['test']  for s in splits.values()])
print(f'\nGlobal: train={len(train_global)}  val={len(val_global)}  test={len(test_global)}')


  central-america        train=    951 val=   203 test=   207
  australia-oceania      train=   2278 val=   488 test=   491
  south-america          train=   7918 val=  1696 test=  1699
  africa                 train=   4108 val=   879 test=   882
  asia                   train=  12908 val=  2765 test=  2768
  europe                 train=  12333 val=  2642 test=  2647
  north-america          train=  17260 val=  3698 test=  3701

Global: train=57756  val=12371  test=12395


## 8. FM adapter: SatlasPretrain

Each FM gets a `Backbone` class with the same interface: `__init__(in_channels, freeze)`, `forward(x) -> (B, feature_dim)`, and a `NAME` attribute. Shared classifier head on top.

**Band handling:** SatlasPretrain S2-MS expects 9 specific Sentinel-2 bands at 512×512. We have 7 bands at 224×224. The first conv is replaced with a 7-channel version whose weights are warm-started by averaging the original 9-channel filter weights. Rest of the backbone stays pretrained.

If this adaptation produces weak linear-probe numbers, the RGB variant (`Sentinel2_SwinB_SI_RGB` with `in_channels=3`, bands `0,1,2`) is the obvious sanity-check alternative.


In [10]:
import torch.nn as nn
import satlaspretrain_models as spm


class SatlasS2Backbone(nn.Module):
    """SatlasPretrain Sentinel-2 multispectral backbone, adapted to 7 input bands."""
    EXPECTED_BANDS_IN = 9
    NAME = 'SatlasPretrain_Sentinel2_SwinB_SI_MS'
    FEATURE_DIM = 1024  # SwinB last-stage channels

    def __init__(self, in_channels=7, freeze=False):
        super().__init__()
        weights_manager = spm.Weights()
        self.backbone = weights_manager.get_pretrained_model(
            model_identifier='Sentinel2_SwinB_SI_MS',
            fpn=False,
        )
        self._adapt_first_conv(in_channels)
        self.feature_dim = self.FEATURE_DIM
        if freeze:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def _adapt_first_conv(self, in_channels):
        for name, module in self.backbone.named_modules():
            if isinstance(module, nn.Conv2d) and module.in_channels == self.EXPECTED_BANDS_IN:
                old = module
                new = nn.Conv2d(
                    in_channels, old.out_channels,
                    kernel_size=old.kernel_size,
                    stride=old.stride,
                    padding=old.padding,
                    bias=(old.bias is not None),
                )
                with torch.no_grad():
                    avg = old.weight.mean(dim=1, keepdim=True)
                    new.weight.copy_(avg.expand(-1, in_channels, -1, -1))
                    if old.bias is not None:
                        new.bias.copy_(old.bias)
                if '.' in name:
                    parent_path, attr = name.rsplit('.', 1)
                    parent = self.backbone
                    for p in parent_path.split('.'):
                        parent = getattr(parent, p)
                    setattr(parent, attr, new)
                else:
                    setattr(self.backbone, name, new)
                print(f'  Adapted first conv: {self.EXPECTED_BANDS_IN} -> {in_channels} channels at "{name}"')
                return
        raise RuntimeError(f'No Conv2d with in_channels={self.EXPECTED_BANDS_IN} found.')

    def forward(self, x):
        feature_maps = self.backbone(x)
        last = feature_maps[-1]               # (B, C, H', W')
        return last.mean(dim=[2, 3])          # global avg pool -> (B, C)


class InfraBenchClassifier(nn.Module):
    """FM backbone + classification head. Shared across FMs."""
    def __init__(self, backbone, num_classes, dropout=0.1):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(backbone.feature_dim, num_classes),
        )
    def forward(self, x):
        return self.head(self.backbone(x))


## 9. Training loop


In [11]:
from torch.optim import AdamW
from collections import Counter, defaultdict
from sklearn.metrics import f1_score, confusion_matrix
import time

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


def _labels_from(dataset):
    """Yield int labels from any (concat/subset/wrapped) dataset."""
    if isinstance(dataset, ConcatDataset):
        for d in dataset.datasets:
            yield from _labels_from(d)
    elif isinstance(dataset, SubsetView):
        for i in dataset.indices:
            yield dataset.base.labels[i]
    else:
        yield from dataset.labels


def compute_class_weights(train_set):
    counts = Counter(_labels_from(train_set))
    total = sum(counts.values())
    weights = torch.zeros(len(CLASS_NAMES))
    for c in range(len(CLASS_NAMES)):
        weights[c] = total / (len(CLASS_NAMES) * counts[c]) if counts.get(c, 0) > 0 else 1.0
    return weights


def collate(batch):
    return {
        'image':  torch.stack([b['image'] for b in batch]),
        'label':  torch.tensor([b['label'] for b in batch], dtype=torch.long),
        'region': [b['region'] for b in batch],
    }


@torch.no_grad()
def evaluate(model, loader, return_per_region=False):
    model.eval()
    all_preds, all_labels, all_regions = [], [], []
    for batch in loader:
        logits = model(batch['image'].to(DEVICE, non_blocking=True))
        preds = logits.argmax(dim=1).cpu().numpy().tolist()
        all_preds.extend(preds)
        all_labels.extend(batch['label'].numpy().tolist())
        all_regions.extend(batch['region'])

    result = {
        'acc': float(np.mean(np.array(all_preds) == np.array(all_labels))),
        'macro_f1': float(f1_score(all_labels, all_preds, average='macro', zero_division=0.0)),
        'per_class_f1': f1_score(all_labels, all_preds, average=None,
                                  labels=list(range(len(CLASS_NAMES))),
                                  zero_division=0.0).tolist(),
        'confusion': confusion_matrix(all_labels, all_preds,
                                       labels=list(range(len(CLASS_NAMES)))).tolist(),
    }
    if return_per_region:
        per = defaultdict(lambda: {'preds': [], 'labels': []})
        for p, l, r in zip(all_preds, all_labels, all_regions):
            per[r]['preds'].append(p)
            per[r]['labels'].append(l)
        result['per_region'] = {
            r: {
                'n': len(d['labels']),
                'macro_f1': float(f1_score(d['labels'], d['preds'], average='macro', zero_division=0.0)),
                'acc': float(np.mean(np.array(d['preds']) == np.array(d['labels']))),
            }
            for r, d in per.items()
        }
    return result


def train_one_run(backbone_factory, train_set, val_set, test_set, *,
                  freeze_backbone, num_epochs, batch_size, lr,
                  weight_decay=1e-4, run_name='run'):
    backbone = backbone_factory(freeze=freeze_backbone)
    model = InfraBenchClassifier(backbone, num_classes=len(CLASS_NAMES)).to(DEVICE)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True,
                              num_workers=4, collate_fn=collate, pin_memory=True)
    val_loader   = DataLoader(val_set,   batch_size=batch_size, shuffle=False,
                              num_workers=4, collate_fn=collate, pin_memory=True)
    test_loader  = DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                              num_workers=4, collate_fn=collate, pin_memory=True)

    weights = compute_class_weights(train_set).to(DEVICE)
    print(f'  Class weights: {weights.cpu().numpy()}')
    criterion = nn.CrossEntropyLoss(weight=weights)

    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = AdamW(trainable, lr=lr, weight_decay=weight_decay)

    history, best_val_f1 = [], -1.0
    for epoch in range(num_epochs):
        model.train()
        t0 = time.time()
        loss_sum, n = 0.0, 0
        for batch in train_loader:
            images = batch['image'].to(DEVICE, non_blocking=True)
            labels = batch['label'].to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * images.size(0)
            n += images.size(0)
        train_loss = loss_sum / max(n, 1)
        val = evaluate(model, val_loader)
        history.append({
            'epoch': epoch + 1, 'train_loss': train_loss,
            'val_acc': val['acc'], 'val_macro_f1': val['macro_f1'],
            'time_s': time.time() - t0,
        })
        marker = ''
        if val['macro_f1'] > best_val_f1:
            best_val_f1 = val['macro_f1']
            marker = ' *'
        print(f'  ep {epoch+1:>3d}  loss={train_loss:.4f}  '
              f'val_acc={val["acc"]:.4f}  val_f1={val["macro_f1"]:.4f}{marker}')

    tail = [h['val_macro_f1'] for h in history[-min(5, len(history)):]]
    test = evaluate(model, test_loader, return_per_region=True)

    return {
        'run_name': run_name,
        'backbone': backbone.NAME,
        'condition': 'linear_probe' if freeze_backbone else 'fine_tune',
        'num_epochs': num_epochs,
        'best_val_f1': best_val_f1,
        'tail_mean_f1': float(np.mean(tail)),
        'tail_std_f1': float(np.std(tail)),
        'history': history,
        'test': test,
    }


Device: cuda


## 10. Run experiments

**Strongly recommended:** set `num_epochs=5` on the linear probe first as a smoke test. If it shows meaningful learning, restart and run the full 30. The fine-tune below is the long pole — don't kick it off until linear probe is confirmed working.


In [ ]:
def build_satlas_backbone(freeze):
    return SatlasS2Backbone(in_channels=7, freeze=freeze)

results = {}

print('=' * 70)
print('RUN 1: Linear probe (frozen backbone)')
print('=' * 70)
results['linear_probe'] = train_one_run(
    backbone_factory=build_satlas_backbone,
    train_set=train_global, val_set=val_global, test_set=test_global,
    freeze_backbone=True,
    num_epochs=LP_EPOCHS, batch_size=LP_BATCH, lr=LP_LR,
    run_name='satlas_s2_linear_probe',
)


RUN 1: Linear probe (frozen backbone)
  Adapted first conv: 9 -> 7 channels at "backbone.backbone.features.0.0"
  Class weights: [4.479293   1.325804   0.49443972]


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/infra_fm_clean/infra_fm_code_only

  ep   1  loss=1.1195  val_acc=0.3987  val_f1=0.3556 *


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/infra_fm_clean/infra_fm_code_only/downstream/common/comm_datasets.py:201: UserWarning: Tile has 9 bands but band_indices=[0, 1, 2, 3, 4, 5, 6] selects only 7. Pass --band-indices explicitly to use all bands (e.g. 0,1,2,3,4,5,6 for sentinel2_ms).
  image = self._load_image(record.path)
/content/infra_fm_clean/infra_fm_code_only/downstream/common/comm_datasets.py:201: UserWarning: Tile has 9 bands but band_indices=[0, 1, 2, 3, 4, 5, 6] selects only 7. Pass --band-indices explicitly to use all bands (e.g. 0,1,

  ep   2  loss=1.1068  val_acc=0.4010  val_f1=0.3601 *


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/content/infra_fm_clean/infra_fm_code_only/downstream/common/comm_datasets.py:201: UserWarning: Tile has 9 bands but band_indices=[0, 1, 2, 3, 4, 5, 6] selects only 7. Pass --band-indices explicitly to use all bands (e.g. 0,1,2,3,4,5,6 for sentinel2_ms).
  image = self._load_image(record.path)
/content/infra_fm_clean/infra_fm_code_only/downstream/common/comm_datasets.py:201: UserWarning: Tile has 9 bands but band_indices=[0, 1, 2, 3, 4, 5, 6] selects only 7. Pass --band-indices explicitly to use all bands (e.g. 0,1,

In [ ]:
print('=' * 70)
print('RUN 2: Full fine-tune')
print('=' * 70)
results['fine_tune'] = train_one_run(
    backbone_factory=build_satlas_backbone,
    train_set=train_global, val_set=val_global, test_set=test_global,
    freeze_backbone=False,
    num_epochs=FT_EPOCHS, batch_size=FT_BATCH, lr=FT_LR,
    run_name='satlas_s2_fine_tune',
)


## 11. Report


In [ ]:
out_path = Path(OUTPUT_DIR) / 'satlas_s2_results_v1.json'
with open(out_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Saved: {out_path}\n')

# Headline table
print(f'{"Condition":<16s}  {"Best val F1":>12s}  {"Tail mean":>10s}  {"Tail std":>10s}  {"Test macro F1":>14s}')
print('-' * 70)
for r in results.values():
    print(f'{r["condition"]:<16s}  {r["best_val_f1"]:>12.4f}  '
          f'{r["tail_mean_f1"]:>10.4f}  {r["tail_std_f1"]:>10.4f}  '
          f'{r["test"]["macro_f1"]:>14.4f}')

print('\nPer-class test F1 (fine-tune):')
for i, c in enumerate(CLASS_NAMES):
    print(f'  {c:<28s} {results["fine_tune"]["test"]["per_class_f1"][i]:.4f}')

print('\nPer-region test macro F1 (fine-tune):')
for region, m in sorted(results['fine_tune']['test']['per_region'].items()):
    print(f'  {region:<22s} n={m["n"]:>6d}  macro_f1={m["macro_f1"]:.4f}')

print('\nTest confusion matrix (fine-tune):')
cm = np.array(results['fine_tune']['test']['confusion'])
print(f'{"":>28s}  ' + '  '.join(f'{c[:6]:>6s}' for c in CLASS_NAMES))
for i, c in enumerate(CLASS_NAMES):
    print(f'{c:>28s}  ' + '  '.join(f'{cm[i][j]:>6d}' for j in range(len(CLASS_NAMES))))


: 

## 12. Next steps

**Add the next FM (Prithvi, Clay, CROMA):**
1. New backbone class with `__init__(in_channels, freeze)`, `forward(x) -> (B, feature_dim)`, `NAME`.
2. Handle that FM's band expectations and normalization quirks in the constructor.
3. Pass `backbone_factory=build_<fm>_backbone` to `train_one_run`. Done.

**S1 fusion (two-tower, v2):**
1. Both `SatlasS2Backbone(in_channels=7)` and `SatlasS1Backbone(in_channels=2)`.
2. Concatenate pooled features: `feature_dim = s2.FEATURE_DIM + s1.FEATURE_DIM`.
3. Change `BAND_INDICES` to `'0,1,2,3,4,5,6,7,8'`, split inside the forward pass.
4. Revisit whether to switch from global to per-band percentile normalization, since SAR and optical value ranges differ a lot.

**Multi-sector extension (once non-substation imagery is fetched):**
- Update `CLASS_NAMES` and `ASSET_TYPE_MAP` to the full ontology. Everything downstream is class-count-agnostic.

**Sanity-check escalation if results look broken:**
1. Confirm pixel values in `[0, 1]` from cell 8 printout.
2. Run linear probe with `LP_EPOCHS=5` first. Tail mean ~0.33 = random = something fundamental is wrong (normalization, band order, label mapping). Tail mean ~0.50+ = signal is there, scale up.
3. If linear probe works but fine-tune is worse: lower `FT_LR` (1e-5 instead of 1e-4) — fine-tuning a pretrained SwinB at 1e-4 can be too aggressive.
4. OOM on fine-tune: drop `FT_BATCH` to 8.
